In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sahilhussain2410/gridlock-round2-3/events_with_spatial_temporal_candidates.parquet
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-3/zone_time_model_table.parquet
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-3/zone_observation_confidence.csv
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-3/zone_id_map_250m.parquet
/kaggle/input/datasets/sahilhussain2410/gridlock-round2-3/zone_support_250m.parquet


# Phase 4 -> Pressure forecasting

## Pressure-Forecasting Baselines

In [2]:
# load only the required columns
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

files = list(
    Path("/kaggle/input").rglob("zone_time_model_table.parquet")
)

if not files:
    raise FileNotFoundError(
        "zone_time_model_table.parquet was not found."
    )

MODEL_TABLE_PATH = files[0]
OUTPUT_DIR = Path("/kaggle/working/parking_phase4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required_columns = [
    "zone_index",
    "time_window",
    "target_time",
    "split_code",
    "incident_count",
    "target_next_hour_pressure",
    "target_next_hour_hotspot",
    "incident_lag_24h",
    "incident_lag_168h",
    "rolling_mean_3h",
    "rolling_mean_24h",
    "same_hour_7d_mean"
]

baseline_data = pd.read_parquet(
    MODEL_TABLE_PATH,
    columns=required_columns
)

baseline_data = baseline_data.loc[
    baseline_data["split_code"].isin([2, 3])
].copy()

print("Loaded:", MODEL_TABLE_PATH)
print("Evaluation shape:", baseline_data.shape)
print(
    "Memory MB:",
    baseline_data.memory_usage(deep=True).sum() / 1024**2
)

Loaded: /kaggle/input/datasets/sahilhussain2410/gridlock-round2-3/zone_time_model_table.parquet
Evaluation shape: (1265344, 12)
Memory MB: 69.9901123046875


In [3]:
# baseline prediction
baseline_data["baseline_zero"] = 0.0

baseline_data["baseline_current_hour"] = (
    baseline_data["incident_count"]
    .astype("float32")
)

baseline_data["baseline_previous_day"] = (
    baseline_data["incident_lag_24h"]
    .fillna(0)
    .clip(lower=0)
    .astype("float32")
)

baseline_data["baseline_previous_week"] = (
    baseline_data["incident_lag_168h"]
    .fillna(0)
    .clip(lower=0)
    .astype("float32")
)

baseline_data["baseline_recent_3h"] = (
    baseline_data["rolling_mean_3h"]
    .fillna(0)
    .clip(lower=0)
    .astype("float32")
)

baseline_data["baseline_recent_24h"] = (
    baseline_data["rolling_mean_24h"]
    .fillna(0)
    .clip(lower=0)
    .astype("float32")
)

baseline_data["baseline_same_hour_7d"] = (
    baseline_data["same_hour_7d_mean"]
    .fillna(0)
    .clip(lower=0)
    .astype("float32")
)

baseline_columns = [
    "baseline_zero",
    "baseline_current_hour",
    "baseline_previous_day",
    "baseline_previous_week",
    "baseline_recent_3h",
    "baseline_recent_24h",
    "baseline_same_hour_7d"
]

display(
    baseline_data[baseline_columns].describe().T
)

,count,mean,std,min,25%,50%,75%,max
baseline_zero,"1,265,344.0000",0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
baseline_current_hour,"1,265,344.0000",0.0606,0.7638,0.0000,0.0000,0.0000,0.0000,78.0000
baseline_previous_day,"1,265,344.0000",0.0605,0.7609,0.0000,0.0000,0.0000,0.0000,78.0000
baseline_previous_week,"1,265,344.0000",0.0593,0.7544,0.0000,0.0000,0.0000,0.0000,78.0000
baseline_recent_3h,"1,265,344.0000",0.0607,0.5266,0.0000,0.0000,0.0000,0.0000,61.6667
baseline_recent_24h,"1,265,344.0000",0.0611,0.2448,0.0000,0.0000,0.0000,0.0000,9.3333
baseline_same_hour_7d,"1,265,344.0000",0.0598,0.3804,0.0000,0.0000,0.0000,0.0000,24.0000


In [4]:
# define evaluation metric
# function calculates ordinary errors and operational ranking metrics.
# Top-K evaluation asks whether the method places the highest future-pressure zones near the top of the enforcement queue
HOTSPOT_THRESHOLD = 8
TOP_K_VALUES = [10, 25, 50]


def prepare_split_matrices(data, split_code, prediction_columns):
    split_data = (
        data.loc[data["split_code"] == split_code]
        .sort_values(["target_time", "zone_index"])
        .reset_index(drop=True)
    )

    zones_per_hour = (
        split_data.groupby("target_time")["zone_index"]
        .nunique()
    )

    assert zones_per_hour.nunique() == 1

    n_hours = split_data["target_time"].nunique()
    n_zones = int(zones_per_hour.iloc[0])

    assert len(split_data) == n_hours * n_zones

    target = (
        split_data["target_next_hour_pressure"]
        .to_numpy(dtype=np.float32)
        .reshape(n_hours, n_zones)
    )

    predictions = {
        col: (
            split_data[col]
            .to_numpy(dtype=np.float32)
            .reshape(n_hours, n_zones)
        )
        for col in prediction_columns
    }

    return target, predictions, n_hours, n_zones


def evaluate_prediction_matrix(
    target,
    prediction,
    model_name,
    split_name,
    top_k_values
):
    target_flat = target.ravel()
    prediction_flat = prediction.ravel()

    active_mask = target_flat > 0

    regression_row = {
        "split": split_name,
        "baseline": model_name,
        "overall_mae": np.mean(
            np.abs(target_flat - prediction_flat)
        ),
        "overall_rmse": np.sqrt(
            np.mean((target_flat - prediction_flat) ** 2)
        ),
        "active_target_mae": np.mean(
            np.abs(
                target_flat[active_mask]
                - prediction_flat[active_mask]
            )
        ),
        "mean_prediction": prediction_flat.mean()
    }

    ranking_rows = []

    for k in top_k_values:
        k = min(k, prediction.shape[1])

        candidate_indices = np.argpartition(
            -prediction,
            kth=k - 1,
            axis=1
        )[:, :k]

        candidate_predictions = np.take_along_axis(
            prediction,
            candidate_indices,
            axis=1
        )

        order = np.argsort(
            -candidate_predictions,
            axis=1
        )

        top_indices = np.take_along_axis(
            candidate_indices,
            order,
            axis=1
        )

        selected_target = np.take_along_axis(
            target,
            top_indices,
            axis=1
        )

        ideal_indices = np.argpartition(
            -target,
            kth=k - 1,
            axis=1
        )[:, :k]

        ideal_target = np.take_along_axis(
            target,
            ideal_indices,
            axis=1
        )

        ideal_target = np.sort(
            ideal_target,
            axis=1
        )[:, ::-1]

        discounts = (
            1 / np.log2(np.arange(2, k + 2))
        ).astype(np.float32)

        dcg = (
            selected_target * discounts
        ).sum(axis=1)

        ideal_dcg = (
            ideal_target * discounts
        ).sum(axis=1)

        valid_ndcg_hours = ideal_dcg > 0

        true_hotspots = target >= HOTSPOT_THRESHOLD
        selected_hotspots = (
            selected_target >= HOTSPOT_THRESHOLD
        )

        hours_with_hotspot = true_hotspots.any(axis=1)

        ranking_rows.append({
            "split": split_name,
            "baseline": model_name,
            "k": k,
            "pressure_capture_at_k": (
                selected_target.sum() / target.sum()
                if target.sum() > 0 else 0
            ),
            "active_zone_recall_at_k": (
                (selected_target > 0).sum()
                / (target > 0).sum()
            ),
            "hotspot_recall_at_k": (
                selected_hotspots.sum()
                / true_hotspots.sum()
                if true_hotspots.sum() > 0 else 0
            ),
            "hotspot_precision_at_k": (
                selected_hotspots.sum()
                / selected_hotspots.size
            ),
            "hotspot_hit_rate_at_k": (
                selected_hotspots[
                    hours_with_hotspot
                ].any(axis=1).mean()
                if hours_with_hotspot.any() else 0
            ),
            "mean_ndcg_at_k": (
                (
                    dcg[valid_ndcg_hours]
                    / ideal_dcg[valid_ndcg_hours]
                ).mean()
                if valid_ndcg_hours.any() else 0
            )
        })

    return regression_row, ranking_rows

In [5]:
# evaluates every baseline on both chronological periods. Validation identifies the strongest simple rule;
# test confirms whether that rule remains stable later

regression_results = []
ranking_results = []

for split_code, split_name in [
    (2, "validation"),
    (3, "test")
]:
    target, predictions, n_hours, n_zones = (
        prepare_split_matrices(
            baseline_data,
            split_code,
            baseline_columns
        )
    )

    print(
        f"{split_name}: "
        f"{n_hours} hours × {n_zones} zones"
    )

    for baseline_name, prediction in predictions.items():
        regression_row, ranking_rows = (
            evaluate_prediction_matrix(
                target=target,
                prediction=prediction,
                model_name=baseline_name,
                split_name=split_name,
                top_k_values=TOP_K_VALUES
            )
        )

        regression_results.append(regression_row)
        ranking_results.extend(ranking_rows)

regression_leaderboard = pd.DataFrame(
    regression_results
)

ranking_leaderboard = pd.DataFrame(
    ranking_results
)

display(
    regression_leaderboard.sort_values(
        ["split", "active_target_mae"]
    )
)

display(
    ranking_leaderboard.loc[
        ranking_leaderboard["k"] == 25
    ].sort_values(
        ["split", "pressure_capture_at_k"],
        ascending=[True, False]
    )
)

validation: 544 hours × 1163 zones
test: 544 hours × 1163 zones


,split,baseline,overall_mae,overall_rmse,active_target_mae,mean_prediction
13,test,baseline_same_hour_7d,0.1026,0.7684,3.1248,0.0596
12,test,baseline_recent_24h,0.1095,0.7511,3.1655,0.0609
11,test,baseline_recent_3h,0.1058,0.8478,3.2520,0.0612
8,test,baseline_current_hour,0.0973,0.9217,3.3748,0.0609
9,test,baseline_previous_day,0.1078,0.9883,3.4427,0.0601
7,test,baseline_zero,0.0607,0.7576,3.4773,0.0000
10,test,baseline_previous_week,0.1077,1.0128,3.4829,0.0589
6,validation,baseline_same_hour_7d,0.1048,0.7981,3.1884,0.0600
5,validation,baseline_recent_24h,0.1113,0.7769,3.2167,0.0614
4,validation,baseline_recent_3h,0.1075,0.8841,3.3001,0.0603


,split,baseline,k,pressure_capture_at_k,active_zone_recall_at_k,hotspot_recall_at_k,hotspot_precision_at_k,hotspot_hit_rate_at_k,mean_ndcg_at_k
37,test,baseline_recent_24h,25,0.2924,0.1792,0.3706,0.0341,0.7097,0.2193
25,test,baseline_current_hour,25,0.2777,0.1946,0.3458,0.0318,0.6569,0.2480
40,test,baseline_same_hour_7d,25,0.2571,0.1573,0.3251,0.0299,0.6716,0.2295
34,test,baseline_recent_3h,25,0.2087,0.1414,0.2532,0.0233,0.5513,0.1755
28,test,baseline_previous_day,25,0.1633,0.1114,0.1989,0.0183,0.4663,0.1348
31,test,baseline_previous_week,25,0.1595,0.1059,0.1917,0.0176,0.4516,0.1359
22,test,baseline_zero,25,0.0102,0.0103,0.0096,0.0009,0.0352,0.0057
16,validation,baseline_recent_24h,25,0.2577,0.1566,0.3333,0.0302,0.7160,0.1849
4,validation,baseline_current_hour,25,0.2571,0.1894,0.3009,0.0273,0.6479,0.2362
19,validation,baseline_same_hour_7d,25,0.2371,0.1488,0.2993,0.0271,0.6479,0.1970


## Catboost Pressure Model

In [6]:
# Load modeling features
import pyarrow.parquet as pq

MODEL_TABLE_PATH = list(
    Path("/kaggle/input").rglob("zone_time_model_table.parquet")
)[0]

OUTPUT_DIR = Path("/kaggle/working/parking_phase4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

schema_columns = pq.ParquetFile(
    MODEL_TABLE_PATH
).schema.names

explicit_features = [
    "zone_index",
    "incident_count",
    "unique_vehicles",
    "named_junction_events",
    "main_road_events",
    "road_crossing_events",
    "footpath_events",
    "large_vehicle_events",
    "multi_offence_events",
    "main_road_share",
    "road_crossing_share",
    "footpath_share",
    "named_junction_share",
    "large_vehicle_share",
    "multi_offence_share",
    "history_hours_available",
    "rolling_std_24h",
    "same_hour_7d_mean",
    "same_hour_7d_active_rate",
    "same_weekhour_4w_mean",
    "consecutive_active_hours_before",
    "short_term_trend",
    "current_burst_z",
    "relative_to_same_hour"
]

dynamic_prefixes = (
    "incident_lag_",
    "unique_vehicles_lag_",
    "past_incidents_",
    "rolling_mean_",
    "active_fraction_"
)

dynamic_features = [
    col for col in schema_columns
    if col.startswith(dynamic_prefixes)
]

feature_columns = list(dict.fromkeys([
    col for col in explicit_features + dynamic_features
    if col in schema_columns
]))

load_columns = list(dict.fromkeys(
    feature_columns + [
        "time_window",
        "target_time",
        "split_code",
        "target_next_hour_pressure"
    ]
))

model_data = pd.read_parquet(
    MODEL_TABLE_PATH,
    columns=load_columns
)

model_data["hour_of_day"] = (
    model_data["time_window"].dt.hour.astype("int8")
)

model_data["day_of_week"] = (
    model_data["time_window"].dt.dayofweek.astype("int8")
)

model_data["month"] = (
    model_data["time_window"].dt.month.astype("int8")
)

model_data["is_weekend"] = (
    model_data["day_of_week"] >= 5
).astype("int8")

feature_columns += [
    "hour_of_day",
    "day_of_week",
    "month",
    "is_weekend"
]

categorical_features = [
    "zone_index",
    "hour_of_day",
    "day_of_week",
    "month"
]

print("Features:", len(feature_columns))
print("Rows:", len(model_data))
print("Memory MB:", model_data.memory_usage(deep=True).sum() / 1024**2)
print(feature_columns)

Features: 49
Rows: 4214712
Memory MB: 807.9121055603027
['zone_index', 'incident_count', 'unique_vehicles', 'named_junction_events', 'main_road_events', 'road_crossing_events', 'footpath_events', 'large_vehicle_events', 'multi_offence_events', 'main_road_share', 'road_crossing_share', 'footpath_share', 'named_junction_share', 'large_vehicle_share', 'multi_offence_share', 'history_hours_available', 'rolling_std_24h', 'same_hour_7d_mean', 'same_hour_7d_active_rate', 'same_weekhour_4w_mean', 'consecutive_active_hours_before', 'short_term_trend', 'current_burst_z', 'relative_to_same_hour', 'incident_lag_1h', 'incident_lag_2h', 'incident_lag_3h', 'incident_lag_6h', 'incident_lag_24h', 'incident_lag_168h', 'unique_vehicles_lag_1h', 'unique_vehicles_lag_24h', 'unique_vehicles_lag_168h', 'past_incidents_3h', 'rolling_mean_3h', 'active_fraction_3h', 'past_incidents_6h', 'rolling_mean_6h', 'active_fraction_6h', 'past_incidents_24h', 'rolling_mean_24h', 'active_fraction_24h', 'past_incidents_168h

In [7]:
# Create the training sample and weights
# Every positive-pressure row is retained. Zero rows are sampled at a 12:1 ratio and
# given inverse-sampling weights so their total statistical contribution remains representative
RANDOM_STATE = 42
NEGATIVE_TO_POSITIVE_RATIO = 12

train_all = model_data.loc[
    model_data["split_code"] == 1
]

positive_index = train_all.index[
    train_all["target_next_hour_pressure"] > 0
].to_numpy()

zero_index = train_all.index[
    train_all["target_next_hour_pressure"] == 0
].to_numpy()

rng = np.random.default_rng(RANDOM_STATE)

n_sampled_zeros = min(
    len(zero_index),
    NEGATIVE_TO_POSITIVE_RATIO * len(positive_index)
)

sampled_zero_index = rng.choice(
    zero_index,
    size=n_sampled_zeros,
    replace=False
)

training_index = np.concatenate([
    positive_index,
    sampled_zero_index
])

rng.shuffle(training_index)

train_sample = model_data.loc[training_index].copy()
validation_data = model_data.loc[
    model_data["split_code"] == 2
].copy()

y_train = train_sample[
    "target_next_hour_pressure"
].to_numpy(dtype=np.float32)

y_validation = validation_data[
    "target_next_hour_pressure"
].to_numpy(dtype=np.float32)

zero_sampling_correction = (
    len(zero_index) / n_sampled_zeros
)

sampling_weight = np.where(
    y_train == 0,
    zero_sampling_correction,
    1.0
)

pressure_weight = 1 + np.sqrt(y_train)

train_weights = (
    sampling_weight * pressure_weight
).astype(np.float32)

y_train_log = np.log1p(y_train)
y_validation_log = np.log1p(y_validation)

X_train = train_sample[feature_columns].copy()
X_validation = validation_data[feature_columns].copy()

for col in categorical_features:
    X_train[col] = X_train[col].astype(str)
    X_validation[col] = X_validation[col].astype(str)

print("All train rows:", len(train_all))
print("Positive rows retained:", len(positive_index))
print("Zero rows sampled:", n_sampled_zeros)
print("Training sample:", len(train_sample))
print("Zero sampling correction:", zero_sampling_correction)

All train rows: 2752821
Positive rows retained: 53119
Zero rows sampled: 637428
Training sample: 690547
Zero sampling correction: 4.235305006996868


In [8]:
# train catboost
from catboost import CatBoostRegressor
import torch

use_gpu = torch.cuda.is_available()

pressure_model = CatBoostRegressor(
    iterations=1200,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    eval_metric="RMSE",
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    task_type="GPU" if use_gpu else "CPU",
    devices="0" if use_gpu else None,
    od_type="Iter",
    od_wait=100,
    allow_writing_files=False,
    verbose=100
)

pressure_model.fit(
    X_train,
    y_train_log,
    sample_weight=train_weights,
    eval_set=(X_validation, y_validation_log),
    cat_features=categorical_features,
    use_best_model=True
)

print("GPU used:", use_gpu)
print("Best iteration:", pressure_model.get_best_iteration())
print("Best validation score:", pressure_model.get_best_score())

0:	learn: 0.3554212	test: 0.1866169	best: 0.1866169 (0)	total: 180ms	remaining: 3m 35s
100:	learn: 0.2953590	test: 0.1854134	best: 0.1748485 (18)	total: 7.27s	remaining: 1m 19s
bestTest = 0.1748484902
bestIteration = 18
Shrink model to first 19 iterations.
GPU used: True
Best iteration: 18
Best validation score: {'learn': {'RMSE': 0.2946692251340516}, 'validation': {'RMSE': 0.17484849020730114}}


In [9]:
# evaluate validation and test rankings
model_results = []
model_ranking_results = []
prediction_outputs = []

for split_code, split_name in [
    (2, "validation"),
    (3, "test")
]:
    split_data = (
        model_data.loc[
            model_data["split_code"] == split_code
        ]
        .sort_values(["target_time", "zone_index"])
        .copy()
    )

    X_split = split_data[feature_columns].copy()

    for col in categorical_features:
        X_split[col] = X_split[col].astype(str)

    predicted_log_pressure = pressure_model.predict(
        X_split
    )

    predicted_pressure = np.clip(
        np.expm1(predicted_log_pressure),
        0,
        None
    ).astype(np.float32)

    zones_per_hour = (
        split_data.groupby("target_time")["zone_index"]
        .nunique()
    )

    assert zones_per_hour.nunique() == 1

    n_hours = split_data["target_time"].nunique()
    n_zones = int(zones_per_hour.iloc[0])

    target_matrix = (
        split_data["target_next_hour_pressure"]
        .to_numpy(dtype=np.float32)
        .reshape(n_hours, n_zones)
    )

    prediction_matrix = predicted_pressure.reshape(
        n_hours,
        n_zones
    )

    regression_row, ranking_rows = (
        evaluate_prediction_matrix(
            target=target_matrix,
            prediction=prediction_matrix,
            model_name="catboost_pressure",
            split_name=split_name,
            top_k_values=[10, 25, 50]
        )
    )

    model_results.append(regression_row)
    model_ranking_results.extend(ranking_rows)

    prediction_outputs.append(
        pd.DataFrame({
            "zone_index": split_data["zone_index"].to_numpy(),
            "target_time": split_data["target_time"].to_numpy(),
            "split": split_name,
            "actual_pressure": (
                split_data["target_next_hour_pressure"]
                .to_numpy()
            ),
            "predicted_pressure": predicted_pressure
        })
    )

model_regression_metrics = pd.DataFrame(model_results)
model_ranking_metrics = pd.DataFrame(
    model_ranking_results
)

display(model_regression_metrics)

display(
    model_ranking_metrics.loc[
        model_ranking_metrics["k"] == 25
    ]
)

,split,baseline,overall_mae,overall_rmse,active_target_mae,mean_prediction
0,validation,catboost_pressure,0.1149,0.7485,3.1573,0.0665
1,test,catboost_pressure,0.1141,0.7233,3.1108,0.0666


,split,baseline,k,pressure_capture_at_k,active_zone_recall_at_k,hotspot_recall_at_k,hotspot_precision_at_k,hotspot_hit_rate_at_k,mean_ndcg_at_k
1,validation,catboost_pressure,25,0.3833,0.2477,0.4753,0.0431,0.8314,0.3379
4,test,catboost_pressure,25,0.3872,0.2536,0.4840,0.0446,0.8123,0.3646


In [10]:
# comparing against baseline and saving
comparison_at_25 = pd.concat([
    ranking_leaderboard.loc[
        (ranking_leaderboard["k"] == 25)
        & ranking_leaderboard["baseline"].isin([
            "baseline_recent_24h",
            "baseline_current_hour",
            "baseline_same_hour_7d"
        ])
    ],
    model_ranking_metrics.loc[
        model_ranking_metrics["k"] == 25
    ]
], ignore_index=True)

display(
    comparison_at_25.sort_values(
        ["split", "pressure_capture_at_k"],
        ascending=[True, False]
    )
)

feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": pressure_model.get_feature_importance()
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

display(feature_importance.head(25))

pressure_predictions = pd.concat(
    prediction_outputs,
    ignore_index=True
)

pressure_model.save_model(
    str(OUTPUT_DIR / "catboost_pressure_model.cbm")
)

pressure_predictions.to_parquet(
    OUTPUT_DIR / "catboost_pressure_predictions.parquet",
    index=False
)

model_regression_metrics.to_csv(
    OUTPUT_DIR / "catboost_regression_metrics.csv",
    index=False
)

model_ranking_metrics.to_csv(
    OUTPUT_DIR / "catboost_ranking_metrics.csv",
    index=False
)

feature_importance.to_csv(
    OUTPUT_DIR / "catboost_feature_importance.csv",
    index=False
)

comparison_at_25.to_csv(
    OUTPUT_DIR / "model_vs_baselines_at_25.csv",
    index=False
)

print("Model and evaluation outputs saved.")

,split,baseline,k,pressure_capture_at_k,active_zone_recall_at_k,hotspot_recall_at_k,hotspot_precision_at_k,hotspot_hit_rate_at_k,mean_ndcg_at_k
7,test,catboost_pressure,25,0.3872,0.2536,0.4840,0.0446,0.8123,0.3646
4,test,baseline_recent_24h,25,0.2924,0.1792,0.3706,0.0341,0.7097,0.2193
3,test,baseline_current_hour,25,0.2777,0.1946,0.3458,0.0318,0.6569,0.2480
5,test,baseline_same_hour_7d,25,0.2571,0.1573,0.3251,0.0299,0.6716,0.2295
6,validation,catboost_pressure,25,0.3833,0.2477,0.4753,0.0431,0.8314,0.3379
1,validation,baseline_recent_24h,25,0.2577,0.1566,0.3333,0.0302,0.7160,0.1849
0,validation,baseline_current_hour,25,0.2571,0.1894,0.3009,0.0273,0.6479,0.2362
2,validation,baseline_same_hour_7d,25,0.2371,0.1488,0.2993,0.0271,0.6479,0.1970


,feature,importance
0,hour_of_day,20.7124
1,incident_count,17.1650
2,unique_vehicles,12.6634
3,past_incidents_168h,10.7425
4,rolling_mean_168h,9.8693
5,same_weekhour_4w_mean,5.8492
6,same_hour_7d_mean,5.1506
7,zone_index,3.1731
8,relative_to_same_hour,2.6369
9,rolling_std_24h,2.4386


Model and evaluation outputs saved.


## Select Model Iteration by Operational Ranking

In [11]:
# This trains 250 trees without RMSE-based early stopping
ranking_candidate_model = CatBoostRegressor(
    iterations=250,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    task_type="GPU" if use_gpu else "CPU",
    devices="0" if use_gpu else None,
    allow_writing_files=False,
    verbose=50
)

ranking_candidate_model.fit(
    X_train,
    y_train_log,
    sample_weight=train_weights,
    cat_features=categorical_features
)

print("Fixed-length candidate model trained.")

0:	learn: 0.3554212	total: 83.2ms	remaining: 20.7s
50:	learn: 0.2992220	total: 3.32s	remaining: 12.9s
100:	learn: 0.2953455	total: 6.44s	remaining: 9.49s
150:	learn: 0.2934252	total: 9.66s	remaining: 6.33s
200:	learn: 0.2919813	total: 12.8s	remaining: 3.12s
249:	learn: 0.2906241	total: 15.9s	remaining: 0us
Fixed-length candidate model trained.


In [12]:
# evaluate tree checkpoint on validation data
validation_eval = (
    validation_data
    .sort_values(["target_time", "zone_index"])
    .copy()
)

X_validation_eval = validation_eval[
    feature_columns
].copy()

for col in categorical_features:
    X_validation_eval[col] = (
        X_validation_eval[col].astype(str)
    )

n_validation_hours = (
    validation_eval["target_time"].nunique()
)

n_validation_zones = (
    validation_eval["zone_index"].nunique()
)

validation_target_matrix = (
    validation_eval["target_next_hour_pressure"]
    .to_numpy(dtype=np.float32)
    .reshape(n_validation_hours, n_validation_zones)
)

checkpoints = [
    10, 20, 30, 50, 75,
    100, 150, 200, 250
]

checkpoint_rows = []

for trees in checkpoints:
    prediction_log = ranking_candidate_model.predict(
        X_validation_eval,
        ntree_end=trees
    )

    prediction_matrix = np.clip(
        np.expm1(prediction_log),
        0,
        None
    ).astype(np.float32).reshape(
        n_validation_hours,
        n_validation_zones
    )

    _, ranking_rows = evaluate_prediction_matrix(
        target=validation_target_matrix,
        prediction=prediction_matrix,
        model_name=f"catboost_{trees}_trees",
        split_name="validation",
        top_k_values=[25]
    )

    row = ranking_rows[0]
    row["trees"] = trees
    checkpoint_rows.append(row)

checkpoint_results = pd.DataFrame(
    checkpoint_rows
).sort_values(
    [
        "pressure_capture_at_k",
        "mean_ndcg_at_k"
    ],
    ascending=False
)

display(checkpoint_results)

,split,baseline,k,pressure_capture_at_k,active_zone_recall_at_k,hotspot_recall_at_k,hotspot_precision_at_k,hotspot_hit_rate_at_k,mean_ndcg_at_k,trees
6,validation,catboost_150_trees,25,0.3938,0.2568,0.4850,0.0440,0.8373,0.3596,150
8,validation,catboost_250_trees,25,0.3925,0.2561,0.4858,0.0440,0.8432,0.3584,250
7,validation,catboost_200_trees,25,0.3921,0.2553,0.4850,0.0440,0.8373,0.3589,200
5,validation,catboost_100_trees,25,0.3906,0.2533,0.4793,0.0435,0.8343,0.3563,100
3,validation,catboost_50_trees,25,0.3896,0.2523,0.4785,0.0434,0.8314,0.3487,50
4,validation,catboost_75_trees,25,0.3884,0.2523,0.4769,0.0432,0.8284,0.3525,75
2,validation,catboost_30_trees,25,0.3858,0.2504,0.4769,0.0432,0.8284,0.3423,30
1,validation,catboost_20_trees,25,0.3826,0.2481,0.4728,0.0429,0.8254,0.3407,20
0,validation,catboost_10_trees,25,0.3680,0.2387,0.4558,0.0413,0.8136,0.3251,10


In [13]:
# select best checkpoint and then use that for test and validation data
BEST_TREES = int(
    checkpoint_results.iloc[0]["trees"]
)

final_pressure_model = ranking_candidate_model

final_pressure_model.shrink(
    ntree_end=BEST_TREES
)

final_metrics = []
final_ranking = []
final_prediction_outputs = []

for split_code, split_name in [
    (2, "validation"),
    (3, "test")
]:
    split_data = (
        model_data.loc[
            model_data["split_code"] == split_code
        ]
        .sort_values(["target_time", "zone_index"])
        .copy()
    )

    X_split = split_data[feature_columns].copy()

    for col in categorical_features:
        X_split[col] = X_split[col].astype(str)

    predicted_pressure = np.clip(
        np.expm1(final_pressure_model.predict(X_split)),
        0,
        None
    ).astype(np.float32)

    n_hours = split_data["target_time"].nunique()
    n_zones = split_data["zone_index"].nunique()

    target_matrix = (
        split_data["target_next_hour_pressure"]
        .to_numpy(dtype=np.float32)
        .reshape(n_hours, n_zones)
    )

    prediction_matrix = predicted_pressure.reshape(
        n_hours,
        n_zones
    )

    regression_row, ranking_rows = (
        evaluate_prediction_matrix(
            target_matrix,
            prediction_matrix,
            "catboost_rank_selected",
            split_name,
            [10, 25, 50]
        )
    )

    final_metrics.append(regression_row)
    final_ranking.extend(ranking_rows)

    final_prediction_outputs.append(
        pd.DataFrame({
            "zone_index": split_data["zone_index"].to_numpy(),
            "target_time": split_data["target_time"].to_numpy(),
            "split": split_name,
            "actual_pressure": (
                split_data["target_next_hour_pressure"]
                .to_numpy()
            ),
            "predicted_pressure": predicted_pressure
        })
    )

final_regression_metrics = pd.DataFrame(final_metrics)
final_ranking_metrics = pd.DataFrame(final_ranking)

print("Selected trees:", BEST_TREES)

display(final_regression_metrics)

display(
    final_ranking_metrics.loc[
        final_ranking_metrics["k"] == 25
    ]
)

Selected trees: 150


,split,baseline,overall_mae,overall_rmse,active_target_mae,mean_prediction
0,validation,catboost_rank_selected,0.1031,0.7291,2.9708,0.0649
1,test,catboost_rank_selected,0.1003,0.7032,2.8921,0.0640


,split,baseline,k,pressure_capture_at_k,active_zone_recall_at_k,hotspot_recall_at_k,hotspot_precision_at_k,hotspot_hit_rate_at_k,mean_ndcg_at_k
1,validation,catboost_rank_selected,25,0.3938,0.2568,0.4850,0.0440,0.8373,0.3596
4,test,catboost_rank_selected,25,0.3993,0.2614,0.5024,0.0462,0.8240,0.3834


In [14]:
#  save the best model, its predictions, metrics
final_feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": (
        final_pressure_model.get_feature_importance()
    )
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

final_predictions = pd.concat(
    final_prediction_outputs,
    ignore_index=True
)

final_pressure_model.save_model(
    str(
        OUTPUT_DIR
        / "catboost_pressure_rank_selected.cbm"
    )
)

final_predictions.to_parquet(
    OUTPUT_DIR
    / "pressure_predictions_rank_selected.parquet",
    index=False
)

checkpoint_results.to_csv(
    OUTPUT_DIR
    / "catboost_iteration_selection.csv",
    index=False
)

final_ranking_metrics.to_csv(
    OUTPUT_DIR
    / "final_pressure_ranking_metrics.csv",
    index=False
)

final_feature_importance.to_csv(
    OUTPUT_DIR
    / "final_pressure_feature_importance.csv",
    index=False
)

display(final_feature_importance.head(20))

print("Final model saved.")

,feature,importance
0,hour_of_day,19.1634
1,incident_count,11.0401
2,unique_vehicles,9.3816
3,zone_index,7.8233
4,rolling_mean_168h,6.7826
5,past_incidents_168h,6.6737
6,same_weekhour_4w_mean,6.0954
7,same_hour_7d_mean,5.6324
8,active_fraction_168h,2.5597
9,rolling_std_24h,2.4348


Final model saved.


## Calibration and Explainability

In [15]:
# validation period is used to learn monotonic mappings from predicted pressure to observed pressure and hotspot probability.
# The test period remains untouched for unbiased evaluation

from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    brier_score_loss
)

validation_predictions = final_predictions.loc[
    final_predictions["split"] == "validation"
].copy()

test_predictions = final_predictions.loc[
    final_predictions["split"] == "test"
].copy()

pressure_calibrator = IsotonicRegression(
    y_min=0,
    out_of_bounds="clip"
)

pressure_calibrator.fit(
    validation_predictions["predicted_pressure"],
    validation_predictions["actual_pressure"]
)

hotspot_calibrator = IsotonicRegression(
    y_min=0,
    y_max=1,
    out_of_bounds="clip"
)

hotspot_calibrator.fit(
    validation_predictions["predicted_pressure"],
    (
        validation_predictions["actual_pressure"] >= 8
    ).astype(int)
)

for data in [
    validation_predictions,
    test_predictions
]:
    data["calibrated_pressure"] = (
        pressure_calibrator.predict(
            data["predicted_pressure"]
        ).astype("float32")
    )

    data["hotspot_probability"] = (
        hotspot_calibrator.predict(
            data["predicted_pressure"]
        ).astype("float32")
    )

calibration_metrics = []

for split_name, data in [
    ("validation", validation_predictions),
    ("test", test_predictions)
]:
    actual = data["actual_pressure"].to_numpy()
    raw = data["predicted_pressure"].to_numpy()
    calibrated = data["calibrated_pressure"].to_numpy()

    hotspot_actual = (actual >= 8).astype(int)

    calibration_metrics.append({
        "split": split_name,
        "mean_actual_pressure": actual.mean(),
        "mean_raw_prediction": raw.mean(),
        "mean_calibrated_prediction": calibrated.mean(),
        "raw_mae": mean_absolute_error(actual, raw),
        "calibrated_mae": mean_absolute_error(
            actual,
            calibrated
        ),
        "raw_rmse": mean_squared_error(
            actual,
            raw
        ) ** 0.5,
        "calibrated_rmse": mean_squared_error(
            actual,
            calibrated
        ) ** 0.5,
        "hotspot_brier_score": brier_score_loss(
            hotspot_actual,
            data["hotspot_probability"]
        )
    })

calibration_metrics = pd.DataFrame(
    calibration_metrics
)

display(calibration_metrics)

,split,mean_actual_pressure,mean_raw_prediction,mean_calibrated_prediction,raw_mae,calibrated_mae,raw_rmse,calibrated_rmse,hotspot_brier_score
0,validation,0.0606,0.0649,0.0606,0.1031,0.0998,0.7291,0.7248,0.0018
1,test,0.0607,0.0640,0.0598,0.1003,0.0976,0.7032,0.7067,0.0018


In [16]:
# Inspect calibration by prediction decile

# This groups test predictions into ten risk bands. 
# A well-calibrated model should show increasing observed pressure and hotspot frequency across the bands.
test_predictions["prediction_decile"] = pd.qcut(
    test_predictions["predicted_pressure"],
    q=10,
    duplicates="drop"
)

calibration_by_decile = (
    test_predictions
    .groupby(
        "prediction_decile",
        observed=True
    )
    .agg(
        rows=("actual_pressure", "size"),
        mean_raw_prediction=(
            "predicted_pressure",
            "mean"
        ),
        mean_calibrated_pressure=(
            "calibrated_pressure",
            "mean"
        ),
        mean_actual_pressure=(
            "actual_pressure",
            "mean"
        ),
        actual_active_rate=(
            "actual_pressure",
            lambda x: (x > 0).mean()
        ),
        actual_hotspot_rate=(
            "actual_pressure",
            lambda x: (x >= 8).mean()
        ),
        mean_hotspot_probability=(
            "hotspot_probability",
            "mean"
        )
    )
    .reset_index()
)

display(calibration_by_decile)

,prediction_decile,rows,mean_raw_prediction,mean_calibrated_pressure,mean_actual_pressure,actual_active_rate,actual_hotspot_rate,mean_hotspot_probability
0,"(-0.001, 0.00177]",253072,0.0001,0.0013,0.0014,0.0006,0.0000,0.0000
1,"(0.00177, 0.0103]",63264,0.0059,0.0071,0.0098,0.0045,0.0003,0.0001
2,"(0.0103, 0.0171]",63268,0.0142,0.0146,0.0150,0.0064,0.0003,0.0003
3,"(0.0171, 0.0239]",63270,0.0202,0.0178,0.0163,0.0073,0.0004,0.0003
4,"(0.0239, 0.0387]",63263,0.0296,0.0260,0.0284,0.0133,0.0005,0.0005
5,"(0.0387, 0.106]",63267,0.0638,0.0657,0.0636,0.0252,0.0015,0.0016
6,"(0.106, 17.191]",63268,0.5060,0.4619,0.4684,0.1155,0.0168,0.0165


In [17]:
# selects the top 25 predicted zones from the final test hour and calculates SHAP contributions. 
# The explanations describe why each zone received its pressure prediction.
from catboost import Pool

latest_target_time = test_predictions[
    "target_time"
].max()

latest_predictions = (
    test_predictions.loc[
        test_predictions["target_time"] == latest_target_time
    ]
    .drop(columns=["prediction_decile"], errors="ignore")
    .nlargest(25, "calibrated_pressure")
    .copy()
)

latest_feature_rows = (
    model_data.loc[
        (model_data["split_code"] == 3)
        & (
            model_data["target_time"]
            == latest_target_time
        )
        & model_data["zone_index"].isin(
            latest_predictions["zone_index"]
        )
    ]
    .sort_values("zone_index")
    .copy()
)

X_explain = latest_feature_rows[
    feature_columns
].copy()

for col in categorical_features:
    X_explain[col] = X_explain[col].astype(str)

explain_pool = Pool(
    X_explain,
    cat_features=categorical_features
)

shap_values = final_pressure_model.get_feature_importance(
    explain_pool,
    type="ShapValues"
)

# Final SHAP column is the model base value.
shap_contributions = shap_values[:, :-1]

explanation_rows = []

for row_position, (_, row) in enumerate(
    latest_feature_rows.iterrows()
):
    contribution_order = np.argsort(
        np.abs(shap_contributions[row_position])
    )[::-1][:5]

    for rank, feature_position in enumerate(
        contribution_order,
        start=1
    ):
        feature = feature_columns[feature_position]

        explanation_rows.append({
            "zone_index": row["zone_index"],
            "target_time": row["target_time"],
            "contribution_rank": rank,
            "feature": feature,
            "feature_value": row[feature],
            "shap_contribution": (
                shap_contributions[
                    row_position,
                    feature_position
                ]
            )
        })

local_explanations = pd.DataFrame(
    explanation_rows
)

display(
    local_explanations.head(25)
)

,zone_index,target_time,contribution_rank,feature,feature_value,shap_contribution
0,192,2024-04-08 23:00:00+05:30,1,hour_of_day,22.0000,-0.1171
1,192,2024-04-08 23:00:00+05:30,2,rolling_std_24h,2.2220,0.0305
2,192,2024-04-08 23:00:00+05:30,3,rolling_mean_168h,0.8036,0.0266
3,192,2024-04-08 23:00:00+05:30,4,past_incidents_168h,135.0000,0.0247
4,192,2024-04-08 23:00:00+05:30,5,same_hour_7d_mean,0.0000,-0.0180
5,194,2024-04-08 23:00:00+05:30,1,hour_of_day,22.0000,-0.0566
6,194,2024-04-08 23:00:00+05:30,2,zone_index,194.0000,0.0162
7,194,2024-04-08 23:00:00+05:30,3,same_hour_7d_mean,0.0000,-0.0141
8,194,2024-04-08 23:00:00+05:30,4,active_fraction_168h,0.0595,0.0139
9,194,2024-04-08 23:00:00+05:30,5,month,4.0000,0.0132


In [18]:
calibrated_predictions = pd.concat(
    [
        validation_predictions.drop(
            columns=["prediction_decile"],
            errors="ignore"
        ),
        test_predictions.drop(
            columns=["prediction_decile"],
            errors="ignore"
        )
    ],
    ignore_index=True
)

latest_priority_preview = (
    latest_predictions.merge(
        local_explanations.loc[
            local_explanations[
                "contribution_rank"
            ] == 1,
            [
                "zone_index",
                "target_time",
                "feature",
                "feature_value",
                "shap_contribution"
            ]
        ],
        on=["zone_index", "target_time"],
        how="left"
    )
    .rename(
        columns={
            "feature": "top_explanation_feature",
            "feature_value": "top_feature_value",
            "shap_contribution":
            "top_feature_contribution"
        }
    )
    .sort_values(
        "calibrated_pressure",
        ascending=False
    )
)

display(
    latest_priority_preview[
        [
            "zone_index",
            "target_time",
            "calibrated_pressure",
            "hotspot_probability",
            "actual_pressure",
            "top_explanation_feature",
            "top_feature_value",
            "top_feature_contribution"
        ]
    ].head(25)
)

calibrated_predictions.to_parquet(
    OUTPUT_DIR
    / "calibrated_pressure_predictions.parquet",
    index=False
)

calibration_metrics.to_csv(
    OUTPUT_DIR / "pressure_calibration_metrics.csv",
    index=False
)

calibration_by_decile.to_csv(
    OUTPUT_DIR / "pressure_calibration_by_decile.csv",
    index=False
)

local_explanations.to_parquet(
    OUTPUT_DIR / "local_pressure_explanations.parquet",
    index=False
)

latest_priority_preview.to_parquet(
    OUTPUT_DIR / "latest_pressure_priority_preview.parquet",
    index=False
)

print("Latest explained target hour:", latest_target_time)
print("Calibration and explanations saved.")

,zone_index,target_time,calibrated_pressure,hotspot_probability,actual_pressure,top_explanation_feature,top_feature_value,top_feature_contribution
0,401,2024-04-08 23:00:00+05:30,0.6709,0.0212,2.0000,same_weekhour_4w_mean,4.0000,0.1765
1,1062,2024-04-08 23:00:00+05:30,0.5693,0.0212,0.0000,zone_index,"1,062.0000",0.1250
2,371,2024-04-08 23:00:00+05:30,0.5527,0.0194,0.0000,incident_count,1.0000,0.0675
3,1052,2024-04-08 23:00:00+05:30,0.1652,0.0052,0.0000,zone_index,"1,052.0000",0.0754
4,400,2024-04-08 23:00:00+05:30,0.1087,0.0029,0.0000,hour_of_day,22.0000,-0.0815
5,377,2024-04-08 23:00:00+05:30,0.0575,0.0012,0.0000,hour_of_day,22.0000,-0.1413
6,407,2024-04-08 23:00:00+05:30,0.0575,0.0012,0.0000,hour_of_day,22.0000,-0.1470
7,717,2024-04-08 23:00:00+05:30,0.0564,0.0012,0.0000,hour_of_day,22.0000,-0.1397
8,435,2024-04-08 23:00:00+05:30,0.0395,0.0006,0.0000,hour_of_day,22.0000,-0.1417
9,731,2024-04-08 23:00:00+05:30,0.0322,0.0006,0.0000,hour_of_day,22.0000,-0.1283


Latest explained target hour: 2024-04-08 23:00:00+05:30
Calibration and explanations saved.
